In [10]:
# Шаг 1. Проверим новую структуру Excel
import openpyxl
from pathlib import Path

input_path = Path("input.xlsx")

wb = openpyxl.load_workbook(input_path)
ws = wb.active

headers = [cell.value for cell in ws[1]]

print("Столбцы:")
for number, header in enumerate(headers, start=1):
    print(number, "->", header)


Столбцы:
1 -> id_r_raspost
2 -> ORG_NAME
3 -> TIP_ORG


In [11]:
import sys
from pathlib import Path
import find_unp_fixed
print("Модуль успешно загружен")

Модуль успешно загружен


In [12]:
names = find_unp_fixed.read_names("input.xlsx", column="ORG_NAME")

In [20]:
LEGAL_FORM = {
    "ооо",
    "чтуп",
    "оао",
    "ип",
    "чуп",
    "одо",
    "чп",
    "чтпуп",
    "зао",
    "чпуп",
    "иооо",
    "уп",
    "руп",
    "пк",
    "пуп",
    "чтсуп",
    "киуп",
    "куп",
    "птчуп",
    "сба засо",
    "мо оао",
    "тп",
    "туп",
    "уз",
    "уо",
    "утк",
    "учтпп",
    "учтп",
    "чптуп",
    "чсуп",
    "чткп",
    "сооо",
    "мукп",
    "филиал зао",
    "филиал оао",
    "частное предприятие",
}

In [15]:
# Шаг 3. Функция определения формы
import re

def find_legal_form(org_name):
    """
    Ищет организационно-правовую форму в начале названия.
    Возвращает:найденную форму в нижнем регистре или None, если форма не найдена.
    """
    if not org_name:
        return None

    name = str(org_name).strip().lower()

    for form in LEGAL_FORM:
        # \b означает границу слова.
        # Поэтому "ип" не сработает внутри другого слова.
        pattern = rf"^{re.escape(form)}\b"

        if re.search(pattern, name, flags=re.IGNORECASE):
            return form

    return None

In [16]:
test_names = [
    'ООО "Ромашка"',
    'ооо "Ромашка"',
    'ОАО "Могилевхлебопродукт"',
    'ип Ковалев Е.В.',
    'ИП Ковалева Ирина Анатольевна',
    'Ковалев Е.В.',
    'РУП "Белдорстрой"',
    'ОДО "Пример"',
]

for name in test_names:
    print(f"{name:45} -> {find_legal_form(name)}")

ООО "Ромашка"                                 -> ооо
ооо "Ромашка"                                 -> ооо
ОАО "Могилевхлебопродукт"                     -> оао
ип Ковалев Е.В.                               -> ип
ИП Ковалева Ирина Анатольевна                 -> ип
Ковалев Е.В.                                  -> None
РУП "Белдорстрой"                             -> руп
ОДО "Пример"                                  -> одо


In [18]:
# Шаг 5. Теперь создаём процедуру обработки Excel - output_tiporg.xlsx
def classify_tip_org(input_file, output_file):
    """
    Читает input.xlsx, определяет TIP_ORG по ORG_NAME и сохраняет ВСЕ исходные данные в новый Excel-файл.
    """
    wb = openpyxl.load_workbook(input_file)
    ws = wb.active

    # Получаем заголовки первой строки
    headers = [cell.value for cell in ws[1]]

    # Ищем номера нужных столбцов
    try:
        org_name_col = headers.index("ORG_NAME") + 1
    except ValueError:
        raise ValueError("Столбец ORG_NAME не найден")

    try:
        tip_org_col = headers.index("TIP_ORG") + 1
    except ValueError:
        raise ValueError("Столбец TIP_ORG не найден")

    print("ORG_NAME -> столбец", org_name_col)
    print("TIP_ORG  -> столбец", tip_org_col)

    # Счётчики для контроля
    count_ul = 0
    count_ip = 0
    count_unknown = 0

    # Обрабатываем все строки, начиная со второй
    for row in range(2, ws.max_row + 1):

        org_name = ws.cell(row=row, column=org_name_col).value

        form = find_legal_form(org_name)

        if form is None:
            # Не смогли определить форму
            # Ничего не ставим автоматически
            ws.cell(row=row, column=tip_org_col).value = ""
            count_unknown += 1

        elif form == "ип":
            # ИП определяем вручную
            ws.cell(row=row, column=tip_org_col).value = ""
            count_ip += 1

        else:
            # Любая другая известная форма = юридическое лицо
            ws.cell(row=row, column=tip_org_col).value = "ЮЛ"
            count_ul += 1

    # Сохраняем новый файл
    wb.save(output_file)

    print()
    print("Обработка завершена")
    print("-------------------")
    print("ЮЛ:", count_ul)
    print("ИП для ручной обработки:", count_ip)
    print("Не определено:", count_unknown)
    print("Всего строк:", ws.max_row - 1)
    print()
    print("Результат:", output_file)

In [21]:
# Шаг 6. Запускаем процедуру обработки Excel
classify_tip_org(
    "input.xlsx",
    "output_tiporg.xlsx"
)

ORG_NAME -> столбец 2
TIP_ORG  -> столбец 3

Обработка завершена
-------------------
ЮЛ: 421
ИП для ручной обработки: 86
Не определено: 231
Всего строк: 738

Результат: output_tiporg.xlsx
